In [1]:
# %load_ext autoreload
# %autoreload 2

In [ ]:
import importlib.util
import os
from collections import Counter

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from dotenv import load_dotenv
from torch.utils.data import DataLoader

from classes import Transformer
from constants import *

In [ ]:
print(torch.cuda.is_available())  # Debería devolver True
print(torch.cuda.get_device_name(0))  # Nombre de tu GPU

In [3]:
df = pd.read_csv(DATOS_PATH, sep="\t", header=None, encoding="utf-8")
df = df.drop(columns=[0, 2])
df = df.rename(columns={1: "spanish", 3: "english"})
df.tail()

,spanish,english
279322,Quiero hablar ucraniano con fluidez.,I want to be fluent in Ukrainian.
279323,Quiero hablar ruso con fluidez.,I want to speak Russian fluently.
279324,Quiero hablar ruso con fluidez.,I want to be fluent in Russian.
279325,Esta música es demasiado psicodélica para mi g...,This music is too psychedelic for my liking.
279326,Esta música es demasiado psicodélica para mi g...,This music is too psychedelic for my tastes.


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 279327 entries, 0 to 279326
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   spanish  279327 non-null  str  
 1   english  279326 non-null  str  
dtypes: str(2)
memory usage: 4.3 MB


### Traductor

In [5]:
spanish = df["spanish"].tolist()
english = df["english"].tolist()

In [6]:
print(spanish[:3])
print(english[:3])

['¡Intentemos algo!', 'Tengo que irme a dormir.', '¿Qué estás haciendo?']
["Let's try something.", 'I have to go to sleep.', 'What are you doing?']


In [7]:
def preprocesar_enunciado(enunciado: str):
    enunciado = str(enunciado)
    enunciado = enunciado.lower().strip()
    # enunciado = enunciado.replace(".", " .")
    # enunciado = enunciado.replace(",", " ,")
    # enunciado = enunciado.replace("¿", " ¿ ")
    # enunciado = enunciado.replace("?", " ? ")
    # enunciado = enunciado.replace("¡", " ¡ ")
    # enunciado = enunciado.replace("!", " ! ")
    enunciado = '<sos> ' + enunciado + ' <eos>'
    return enunciado

In [8]:
enunciado = "¡Hola! ¿Cómo estás?"
print(preprocesar_enunciado(enunciado))

<sos> ¡hola! ¿cómo estás? <eos>


In [9]:
spanish_preprocesado = [preprocesar_enunciado(s) for s in spanish]
english_preprocesado = [preprocesar_enunciado(e) for e in english]

In [10]:
print(spanish_preprocesado[0:3])
print(english_preprocesado[0:3])

['<sos> ¡intentemos algo! <eos>', '<sos> tengo que irme a dormir. <eos>', '<sos> ¿qué estás haciendo? <eos>']
["<sos> let's try something. <eos>", '<sos> i have to go to sleep. <eos>', '<sos> what are you doing? <eos>']


In [11]:
def construir_palabras(enunciados):
    palabras = [palabra for enunciado in enunciados for palabra in enunciado.split(" ")]
    palabras_count = Counter(palabras)
    palabras_ordenadas = sorted(palabras_count.items(), key=lambda x: x[1], reverse=True)
    palabra_idx = {palabra: idx for idx, (palabra, _) in enumerate(palabras_ordenadas, 2)}
    palabra_idx["<pad>"] = 0
    palabra_idx["<unk>"] = 1
    
    idx_palabra = {idx: palabra for palabra, idx in palabra_idx.items()}
    return palabra_idx, idx_palabra

In [12]:
english_palabra_idx, english_idx_palabra = construir_palabras(english_preprocesado)
spanish_palabra_idx, spanish_idx_palabra = construir_palabras(spanish_preprocesado)

In [13]:
spanish_vocab_size = len(spanish_palabra_idx)
english_vocab_size = len(english_palabra_idx)

spanish_vocab_size, english_vocab_size

(90647, 57709)

### Dataset

In [ ]:


# Importar dataset
spec = importlib.util.spec_from_file_location("dataset", "/content/transformer/dataset.py")
dataset_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dataset_module)

Traductor = dataset_module.Traductor
collate_fn = dataset_module.collate_fn
train = dataset_module.train

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\content/transformer/dataset.py'

In [ ]:
dataset = Traductor(spanish_preprocesado, english_preprocesado, spanish_palabra_idx, english_palabra_idx)
dataset[279325]

In [ ]:
english_batch, spanish_batch = dataset[279325]
type(english_batch), type(spanish_batch)


In [ ]:
english_batch, spanish_batch

In [ ]:
english_batch = english_batch.unsqueeze(0)
spanish_batch = spanish_batch.unsqueeze(0)

english_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in english_batch] # el detach para que no calcule ningun gradiente
spanish_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in spanish_batch]

In [ ]:
english_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in english_batch] # el detach para que no calcule ningun gradiente
spanish_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in spanish_batch]

In [ ]:
english_batch, spanish_batch

In [ ]:
# Agrego pading

padded_english_tokens = torch.nn.utils.rnn.pad_sequence(english_batch, batch_first=True, padding_value=0)
padded_spanish_tokens = torch.nn.utils.rnn.pad_sequence(spanish_batch, batch_first=True, padding_value=0)

In [ ]:
padded_english_tokens, padded_spanish_tokens

### Entrenamiento

In [ ]:
dataset

In [ ]:
BATCH_SIZE = 64

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

In [ ]:


model = Transformer(
    d_model=512, 
    num_heads=8, 
    d_ff=2048,
    num_layers=6,
    input_vocab_size=spanish_vocab_size, 
    target_vocab_size=english_vocab_size, 
    max_len=MAX_SEQ_LEN, 
    dropout=0.1
)

In [ ]:


model.to(DEVICE)

In [ ]:


criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
def sentence_to_indices(sentence, word2idx):
    return [word2idx.get(word, word2idx['<unk>']) for word in sentence.split()]

def indices_to_sentence(indices, idx2word):
    return ' '.join([idx2word[idx] for idx in indices if idx in idx2word and idx2word[idx] != '<pad>'])

def translate_sentence(model, sentence, spanish_palabra_idx, english_idx_palabra, max_len=MAX_SEQ_LEN, device='cpu'):
    model.eval()
    sentence = preprocesar_enunciado(sentence)
    input_indices = sentence_to_indices(sentence, spanish_palabra_idx)
    input_tensor = torch.tensor(input_indices).unsqueeze(0).to(device)

    # Initialize the target tensor with <sos> token
    tgt_indices = [english_palabra_idx['<sos>']]
    tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_len):
            output = model(input_tensor, tgt_tensor)
            output = output.squeeze(0)
            next_token = output.argmax(dim=-1)[-1].item()
            tgt_indices.append(next_token)
            tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)
            if next_token == english_palabra_idx['<eos>']:
                break

    return indices_to_sentence(tgt_indices, english_idx_palabra)


In [ ]:
def evaluate_translations(model, sentences, spanish_palabra_idx, english_idx_palabra, max_len=MAX_SEQ_LEN, device='cpu'):
    for sentence in sentences:
        translation = translate_sentence(model, sentence, spanish_palabra_idx, english_idx_palabra, max_len, device)
        print(f'Input sentence: {sentence}')
        print(f'Traducción: {translation}')
        print()

# Example sentences to test the translator
test_sentences = [
    "Hola, cómo estás?",
    "Yo soy una IA",
    "La inteligencia artificial es genial",
    "Viva Argentina!"
]

# Assuming the model is trained and loaded
# Set the device to 'cpu' or 'cuda' as needed
#model = model.to(DEVICE)

# Evaluate translations
#evaluate_translations(model, test_sentences, spanish_palabra_idx, english_idx_palabra, max_len=MAX_SEQ_LEN, device=device)

In [ ]:
train(model, dataloader, criterion, optimizer, epochs=5)